# Clase 056 — Selección y entrenamiento de modelo

Entrenamos baselines, los comparamos con **cross-validation** (no un solo split), leemos
**learning curves** para diagnosticar bias/varianza y elegimos el candidato para fine-tuning.
Persistimos el ganador con `joblib`.

Requiere: `numpy`, `pandas`, `scikit-learn`, `joblib`, `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import time, joblib, tempfile, os
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, learning_curve
from sklearn.metrics import root_mean_squared_error

np.random.seed(42)
data = load_diabetes()
X, y = data.data, data.target
print('diabetes:', X.shape, '| target rango', y.min(), '-', y.max())

## 1. Baseline LinearRegression: RMSE en train

Punto de comparación honesto. Si el lineal ya alcanza, no hace falta un bosque.

In [ ]:
lin = LinearRegression().fit(X, y)
rmse_lin_train = root_mean_squared_error(y, lin.predict(X))
print('primeras 5 preds:', np.round(lin.predict(X[:5]), 1))
print('reales          :', y[:5])
print(f'RMSE train (lineal): {rmse_lin_train:.2f}')

## 2. El árbol que memoriza: RMSE ~ 0 en train

Un `DecisionTreeRegressor` sin podar llega a RMSE 0 en train... porque memorizó. Eso NO lo
hace bueno: hay que medir en datos no vistos.

In [ ]:
tree = DecisionTreeRegressor(random_state=42).fit(X, y)
rmse_tree_train = root_mean_squared_error(y, tree.predict(X))
print(f'RMSE train (arbol): {rmse_tree_train:.4f}')
assert rmse_tree_train < 1.0, 'el arbol sin podar memoriza el train (RMSE ~ 0)'
print('RMSE 0 en train = overfitting, no calidad')

## 3. Cross-validation honesto (cv=10)

`cross_val_score` con `neg_root_mean_squared_error` estima el error de generalización sin
tocar un test set. Recordá negar el signo.

In [ ]:
def cv_rmse(model, cv=10):
    s = cross_val_score(model, X, y, scoring='neg_root_mean_squared_error', cv=cv)
    return -s

rmse_lin_cv = cv_rmse(LinearRegression())
rmse_tree_cv = cv_rmse(DecisionTreeRegressor(random_state=42))
print(f'lineal  CV RMSE: {rmse_lin_cv.mean():.2f} +/- {rmse_lin_cv.std():.2f}')
print(f'arbol   CV RMSE: {rmse_tree_cv.mean():.2f} +/- {rmse_tree_cv.std():.2f}')
print('el arbol que daba 0 en train generaliza peor que el lineal')

## 4. Random Forest + tabla comparativa

El bosque suele ganarle a los baselines en tabular. Medimos tiempo de fit y RMSE-CV.

In [ ]:
def fit_time(model):
    t0 = time.perf_counter(); model.fit(X, y); return time.perf_counter() - t0

rmse_rf_cv = cv_rmse(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
tabla = pd.DataFrame({
    'modelo': ['LinearRegression', 'DecisionTree', 'RandomForest'],
    'RMSE_cv_mean': [rmse_lin_cv.mean(), rmse_tree_cv.mean(), rmse_rf_cv.mean()],
    'RMSE_cv_std':  [rmse_lin_cv.std(), rmse_tree_cv.std(), rmse_rf_cv.std()],
    'fit_seg': [fit_time(LinearRegression()),
                fit_time(DecisionTreeRegressor(random_state=42)),
                fit_time(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))],
}).round(3)
print(tabla.to_string(index=False))

## 5. Learning curve del elegido + persistencia con joblib

Diagnosticamos bias/varianza del RandomForest y lo serializamos con `joblib.dump`.

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
sizes, tr, va = learning_curve(rf, X, y, cv=5,
                               scoring='neg_root_mean_squared_error',
                               train_sizes=np.linspace(0.1, 1.0, 5))

rf.fit(X, y)
path = os.path.join(tempfile.gettempdir(), 'modelo_baseline.pkl')
joblib.dump(rf, path)
reload = joblib.load(path)
assert np.allclose(reload.predict(X[:5]), rf.predict(X[:5])), 'debe predecir identico'
print(f'modelo serializado en {path} y recargado OK')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, -tr.mean(axis=1), 'o-', label='train RMSE')
ax.plot(sizes, -va.mean(axis=1), 's-', label='validation RMSE')
ax.set_xlabel('tamano de train'); ax.set_ylabel('RMSE')
ax.set_title('Learning curve - RandomForest'); ax.legend()
plt.tight_layout(); plt.show()

## Ejercicios

1. Agregá `GradientBoostingRegressor` a la tabla comparativa con `cv=10`. ¿Le gana al
   RandomForest? ¿A qué costo de tiempo?
2. Diagnosticá la learning curve: ¿la brecha train-val se cierra? ¿Es alta varianza, alto bias
   o convergencia? ¿Qué harías a continuación?
3. Corré el mismo experimento envolviendo cada modelo en un `Pipeline` con `StandardScaler`.
   ¿Cambia algo para el lineal? ¿Y para el RandomForest? ¿Por qué?
4. Escribí una "model card" mínima en markdown para el modelo elegido: dataset, métricas CV,
   hiperparámetros, fecha y limitaciones.

## Conclusiones

- El RMSE de train no sirve para rankear modelos: el árbol da 0 y generaliza mal.
- CV (cv=10) da media ± desvío: mide generalización sin tocar el test.
- La learning curve distingue si el cuello es datos (alta varianza) o modelo (alto bias).
- `joblib.dump/load` persiste el candidato para retomarlo en fine-tuning.

## ✅ Soluciones de los ejercicios

El README plantea los ejercicios sobre California Housing (con RMSE en miles de USD), pero ese dataset requiere descarga. Los resolvemos sobre `load_diabetes` (offline, ya cargado como `X`, `y`): la mecánica y las conclusiones son idénticas, solo cambian las unidades. `n_jobs=1`.

**Ej. 1 — Baseline lineal.** `LinearRegression`, predicción sobre 5 ejemplos vs reales y RMSE en train. (En California Housing rondaría ~68k USD; en diabetes es del orden de la escala del target.)

In [ ]:

import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

X, y = load_diabetes(return_X_y=True)
lin = LinearRegression().fit(X, y)
print("preds:", np.round(lin.predict(X[:5]), 1))
print("reales:", y[:5])
rmse = root_mean_squared_error(y, lin.predict(X))
print(f"RMSE train (lineal): {rmse:.2f}")
assert rmse > 0

**Ej. 2 — Árbol que memoriza.** Un `DecisionTreeRegressor` sin podar da RMSE ~0 en train. Eso es *overfitting*, no calidad: memorizó los ejemplos y no aprendió a generalizar.

In [ ]:

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

tree = DecisionTreeRegressor(random_state=42).fit(X, y)
rmse_tree = root_mean_squared_error(y, tree.predict(X))
print(f"RMSE train (arbol sin podar): {rmse_tree:.4f}")
assert rmse_tree < 1.0, "el arbol memoriza el train -> RMSE ~ 0"
print("RMSE 0 en train NO significa buen modelo: hay que medir en datos no vistos (CV)")

**Ej. 3 — Cross-validation honesto (cv=10).** El árbol que daba 0 en train generaliza peor que el lineal cuando lo medimos con CV.

In [ ]:

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

def cv_rmse(model):
    return -cross_val_score(model, X, y, scoring="neg_root_mean_squared_error", cv=10)

lin_cv = cv_rmse(LinearRegression())
tree_cv = cv_rmse(DecisionTreeRegressor(random_state=42))
print(f"lineal CV RMSE: {lin_cv.mean():.2f} +/- {lin_cv.std():.2f}")
print(f"arbol  CV RMSE: {tree_cv.mean():.2f} +/- {tree_cv.std():.2f}")
assert tree_cv.mean() > lin_cv.mean(), "el arbol que memoriza generaliza peor"

**Ej. 4 — Random Forest + tabla comparativa.** Bagging de árboles reduce la varianza y suele mejorar al baseline. Tabla con media ± desvío del RMSE-CV.

In [ ]:

import numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

def cv_rmse(model):
    return -cross_val_score(model, X, y, scoring="neg_root_mean_squared_error", cv=10)

rows = []
for name, m in [("LinearRegression", LinearRegression()),
                ("DecisionTree", DecisionTreeRegressor(random_state=42)),
                ("RandomForest", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=1))]:
    s = cv_rmse(m)
    rows.append({"modelo": name, "RMSE_mean": round(s.mean(), 2), "RMSE_std": round(s.std(), 2)})
tabla = pd.DataFrame(rows)
print(tabla.to_string(index=False))
mejor = tabla.loc[tabla["RMSE_mean"].idxmin(), "modelo"]
print(f"mejor por RMSE-CV: {mejor}")
assert tabla["RMSE_mean"].min() > 0

**Ej. 5 — Learning curve.** Diagnóstico bias/varianza del Random Forest: si train y validación no convergen y la brecha es grande → alta varianza (más datos ayudan); si convergen alto → alto bias/error irreducible.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import learning_curve

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=1)
sizes, tr, va = learning_curve(rf, X, y, cv=5, scoring="neg_root_mean_squared_error",
                               train_sizes=np.linspace(0.1, 1.0, 5))
tr_m, va_m = -tr.mean(1), -va.mean(1)
brecha = va_m[-1] - tr_m[-1]
print(f"RMSE train final={tr_m[-1]:.1f} | val final={va_m[-1]:.1f} | brecha={brecha:.1f}")
diag = "ALTA VARIANZA (brecha grande: mas datos ayudarian)" if brecha > 0.3 * va_m[-1] else "convergencia/alto bias"
print("diagnostico:", diag)
plt.plot(sizes, tr_m, "o-", label="train"); plt.plot(sizes, va_m, "s-", label="validation")
plt.xlabel("tamano de train"); plt.ylabel("RMSE"); plt.legend(); plt.title("Learning curve - RF"); plt.show()
assert va_m[-1] >= tr_m[-1], "la validacion no puede ser mejor que el train en promedio"